# OLMo overnight: routing, content, geometry and forgetting

Complete code for a GH200, retaining the original FP32 source experiment. Run **Setup**, **Configure**, then **Run / resume**. The experiment runs in the background. Rerun **Refresh status** or **Show results** whenever you want; nothing continuously prints.

The default session has an **11.5-hour budget**, including a 6-minute reserve for reports/export. Results are saved incrementally. A session may finish only part of the queue; resume grants another session with the same settings. The results ZIP is generated automatically.

Keep `structure_study.py` and `forgetting_mechanisms.py` beside this notebook. See README.md for what is measured and how to interpret the interventions.

## Setup

In [ ]:
from pathlib import Path
import importlib, json
import structure_study as study
importlib.reload(study)
SESSION_FILE = Path.cwd() / "overnight_session.json"
saved = json.loads(SESSION_FILE.read_text()) if SESSION_FILE.exists() else {}
search = [Path("/home/ubuntu/1/runs/olmo_association_v1")]
search += [p / "runs" / "olmo_association_v1" for p in [Path.cwd(), Path.cwd().parent, Path.home() / "1"]]
if saved.get("source"):
    search.insert(0, Path(saved["source"]))
found = list(dict.fromkeys(p.resolve() for p in search if (p / "config.json").exists()))
SOURCE = found[0] if found else None
print("Original source campaigns:", *found, sep="\n")
# If needed, replace this with the ORIGINAL campaign, not a measurement ZIP:
# SOURCE = Path("/your/original/olmo_association_v1")

## Configure

All available saved source seeds are selected automatically. The primary event is step 21; step 20 is its control, followed by 47/46 and additional seeds. This is a priority queue, not a guarantee that every stage will fit in one night. The actual hardware is recorded at launch. No model downloads beyond the original pinned OLMo/tokenizer are introduced.

In [ ]:
if SOURCE is None:
    raise FileNotFoundError("Set SOURCE to the original campaign containing config.json and seed folders.")
config = study.Config(**study.read(SOURCE / "config.json"))
seeds = [seed for seed in config.seeds
         if (SOURCE / f"seed{seed}" / "anchor.pt").exists()
         and (SOURCE / f"seed{seed}" / "data.json").exists()
         and list((SOURCE / f"seed{seed}").glob("fork*/fork.pt"))]
if not seeds:
    raise FileNotFoundError("No complete source seed folders found.")
OUTPUT = Path(saved["output"]) if saved.get("source") == str(SOURCE) else SOURCE.parent / "olmo_overnight_v1"
SETTINGS = study.default_settings(SOURCE, OUTPUT)
SETTINGS.update(device="cuda:0", threads=8, seeds=seeds)
SETTINGS["overnight"].update(
    hours=11.5,                  # Includes the report/export reserve
    reserve_minutes=6,
    events=[21, 20, 47, 46],     # Actual B-update numbers; source must include preceding forks
    minimum_free_gb=8.0,
    export_arrays=False,        # Raw arrays stay in the results folder; share ZIP holds tables/figures
)
# Default: dense grid across the update plus 0.005 spacing from 0.50 through 0.65.
# All layers and all entities are measured. Keep these defaults for the first overnight run.
study.validate(SETTINGS)
print("Seeds:", seeds)
print("Session hours:", SETTINGS["overnight"]["hours"])
print("Initial path fractions:", len(SETTINGS["overnight"]["alphas"]))
print("Planned jobs:", len(study.night_plan(SETTINGS)))
print("Results:", SETTINGS["output"])

## Run / resume

Safe to rerun: if already active, it reports status. If stopped or budget-paused, it continues unfinished measurements. Changed code/settings automatically select a fresh output folder. The session file remembers the active output path across notebook-kernel restarts.

In [ ]:
OUTPUT = study.launch(SETTINGS)
study.atomic_json(SETTINGS, SESSION_FILE)

## Refresh status

Rerun this cell manually. A heartbeat shows liveness; the phase, fraction, layer and condition show what is being measured. `budget_paused` means the session ended with saved, resumable partial results.

In [ ]:
study.print_status(SETTINGS["output"])

## Show results

Prints compact numbers and displays saved plots. Detailed per-entity, parameter, channel and intervention tables remain on disk.

In [ ]:
study.show(SETTINGS["output"], full=False)

## Stop

Set STOP=True and run this cell to stop the tracked experiment. Completed units are preserved. Use Run / resume to continue, or Restart below to start fresh.

In [ ]:
STOP = False
if STOP:
    print(study.stop(SETTINGS["output"]))
    STOP = False

## Restart from scratch

Set RESTART=True and run this cell. It stops the tracked run and starts a fresh results directory. Previous results remain available.

In [ ]:
RESTART = False
if RESTART:
    OUTPUT = study.restart(SETTINGS)
    study.atomic_json(SETTINGS, SESSION_FILE)
    RESTART = False

## Find / regenerate the results ZIP

An archive is created automatically at session end. This cell shows a download link. To export current partial results, set EXPORT_NOW=True. Include raw arrays only if you want a much larger archive; they remain available in the results directory regardless.

In [ ]:
EXPORT_NOW = False
INCLUDE_RAW_ARRAYS = False
if EXPORT_NOW:
    study.night_report(SETTINGS["output"])
    archive = study.night_export(SETTINGS["output"], include_arrays=INCLUDE_RAW_ARRAYS)
else:
    archive = Path(SETTINGS["output"]) / "results_share.zip"
print("Results directory:", SETTINGS["output"])
print("Results ZIP:", archive)
if archive.exists():
    # Put a small link in the notebook folder; the large ZIP remains at its original location.
    link = Path.cwd() / "overnight_results.zip"
    if link.is_symlink():
        link.unlink()
    if not link.exists():
        try:
            link.symlink_to(archive)
        except OSError:
            pass
    from IPython.display import FileLink, display
    if link.is_symlink() and link.resolve() == archive.resolve():
        display(FileLink("overnight_results.zip"))
    print("If Jupyter blocks the link, open the printed results directory and download results_share.zip there.")
else:
    print("No archive yet. It will be created at session end, or set EXPORT_NOW=True.")

## Optional: inspect recent logs

In [ ]:
SHOW_LOG = False
if SHOW_LOG:
    root = Path(SETTINGS["output"])
    status = study.status(root)
    log = root / status["job"] / "worker.log" if status.get("job") else root / "launcher.log"
    print(log)
    if log.exists():
        print(log.read_text(errors="replace")[-10000:])